# 聚类算法在模式识别与群体划分中的应用

## 1. 项目背景

聚类算法是机器学习中无监督学习的重要分支，其核心是在没有标签指引的情况下，通过衡量数据点之间的相似性（如距离、密度等），将数据集自动划分为若干个”簇“。  
同一簇内的样本具有较高的相似度，不同簇的样本则差异显著。这种算法无需人工预先定义类别，完全依赖数据自身的分布特征完成分组，常见的有快速高效的K-Means、能体现层次关系的层次聚类、可识别任意形状簇的DBSCAN等。  
它广泛应用于客户分群、文本主题挖掘、图像分割等场景，帮助人们从海量数据中发现隐藏的结构与规律。  
为了充分掌握聚类算法以及其应用，本项目使用商场客户数据集进行消费群体划分。使用K-means和DBSCAN算法对客户进行分组，从而识别不同消费群体的特征以及为商场精准营销提供数据支持。最后，比较不同聚类算法的效果。  
通过本次实训，将深入理解聚类算法的基本原理和操作。通过应用该模型，可以有效地获取数据特征并加以划分分组，从而为后续决策提供数据支撑，提高效率与准确性。

In [ ]:
ls /home/jovyan/work/datasets/688b1e924e03dbf50518a12d-momodel

## 2. 数据处理

在这个项目中，使用商场客户数据集Mall_Customers进行测试。
Mall_Customers数据集包含200个样本，特征包括：  


*   CustomerID：客户ID
*   Gender：性别
*   Age：年龄
*   Annual Income(k$)：年收入(千美元)
*   Spending Score（1-100)：消费评分（1-100分）





### 2.1 导入所需库

首先进行环境的配置，导入所需的科学计算库，再进行下一步的数据分析。

In [ ]:
# 导入所需库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D

### 2.2 数据预览

加载商场客户数据集，并输出数据集的各项参数。首先展示数据集的前五行数据：

In [ ]:
# 加载数据
df = pd.read_csv('/home/jovyan/work/datasets/688b1e924e03dbf50518a12d-momodel/Mall_Customers.csv')
print(df.head())

我们需要再进一步了解数据集的基本信息，包括数据集的形状，数据集的缺失值统计情况，以及数据类型：

In [ ]:
# 展示数据集的各项信息，包括数据集的数据量和每个数据的维度，缺失值的统计，以及数据集每个数据的数据类型
print("\n数据基本信息：")
print(f"数据集形状：{df.shape}")
print(f"缺失值统计：\n{df.isnull().sum()}")
print(f"数据类型：\n{df.dtypes}")

从中分析可知，数据集一共有200组数据，每组数据有五个值。在这五个值中，除了性别为object型，其余都为int整型。数据集也没有缺失值。

### 2.3 数据探索与可视化

数据探索与可视化是聚类分析的关键前期工作，其主要目的是：  


1.   **理解数据分布**：通过统计描述和直方图分析特征的集中趋势、离散程度和分布形态
2.   **发现特征关系**：通过散点图等可视化技术揭示特征间的相关性或聚类趋势
3.   **识别异常值**：检测可能影响聚类结果的异常数值点
4.   **特征选择依据**：为后续聚类分析确定最有效的特征组合  

在本项目中，重点关注三个关键特征：


*   **年龄**：分析客户年龄分布，识别主要年龄段
*   **年收入**：考察客户收入水平及其分布
*   **消费评分**：评估客户消费活跃度  

通过多维可视化，特别关注：


*   收入与消费评分的相关关系
*   年龄与消费行为的关系


In [ ]:
# 数值特征分布可视化，创建一个1行3列的子图布局
# 用于同时展示三个特征的分布情况，方便对比分析
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 绘制年龄分布直方图。直方图展示数据分布区间，核密度曲线展示数据分布的概率密度
sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('年龄分布')

# 绘制年收入分布直方图，使用不同颜色区分
sns.histplot(df['Annual Income (k$)'], bins=20, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('年收入分布')

# 绘制消费评分分布直方图，使用不同颜色区分
sns.histplot(df['Spending Score (1-100)'], bins=20, kde=True, ax=axes[2], color='lightgreen')
axes[2].set_title('消费评分分布')

# 自动调整子图间距，避免重叠
plt.tight_layout()
plt.show()

# 收入-消费评分关系
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)',hue='Gender', palette='viridis', s=80, alpha=0.8)
plt.title('收入与消费的关系')
plt.show()

# 年龄与消费评分关系
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)', hue='Gender', palette='coolwarm', s=80, alpha=0.8)
plt.title('年龄与消费评分关系')
plt.show()

从图表来看，收入与消费关系图里，不同性别的消费评分随年收入变化无明显单一规律，但整体收入高时消费评分有多样分布  
年龄分布、年收入分布、消费评分分布各有特征，年龄集中在30 - 40岁左右，年收入在60 - 80（k$）区间相对突出，消费评分有特定集中范围  
年龄与消费评分关系图中，不同性别在各年龄段的消费评分分散，无显著因年龄和性别导致的消费评分规律，说明消费评分受年龄、性别影响复杂，更多受其他因素左右 。 

### 2.4 数据预处理

在数据预处理阶段，我们进行了系统性的特征选择和标准化处理，这是聚类分析成功的关键前提    
首先，我们基于业务理解选择了三个核心特征：Annual Income（k$）（年收入）、Speeding Score（1-100)（消费评分）Age（年龄）  
这些特征能够全面反映客户的消费能力和行为模式，是群体划分最相关的维度。随后，我们对这些特征进行了标准化处理，使用StandardScaler将数据转换为均值0、标准差为1的分布  
这一步骤至关重要，因为聚类算法（特别是基于距离的K-means和DBSCAN）对特征的量纲非常敏感。标准化确保了不同量级的特征（如年龄范围20-70岁和收入范围15-137千美元）具有同等的重要性权重，避免了某些特征因其数值范围较大而主导聚类结果的问题  
最终得到的标准化数据集为后续的聚类分析奠定了公平、可靠的基础。

In [ ]:
# 选择特征并标准化
features = ['Annual Income (k$)', 'Spending Score (1-100)', 'Age']
# 从DataFrame中提取选中的特征列，形成特征矩阵X
X = df[features]

# 标准化数据，目的是将不同量级的特征转换到同一尺度，避免某一特征对结果产生过大影响
scaler = StandardScaler()
# 对特征矩阵X进行标准化处理
X_scaled = scaler.fit_transform(X)

print("标准化后数据前五行：")
print(X_scaled[:5])

从输出结果看，经处理后的数据呈现标准化分布形态，便于后续基于这些特征开展聚类分析，为挖掘数据潜在规律筑牢基础。

## 3. 聚类分析

在聚类分析部分，我们选择两种聚类算法分别进行分析。

### 3.1 K-Means聚类

**K-Means聚类算法简介**
K-Means是一种最常用的无监督学习算法，用于将数据自动划分为K个有意义的组别（簇）。其核心思想是最小化簇内平方和。  
**算法核心**

1.   **初始化中心点**：  
        *   随机选择K个数据点作为初始聚类中心（质心）
2.   **分配阶段**
        *   计算每个数据点到所有质心的距离（通常用欧式距离）
        *   将每个点分配到最近的质心所在簇
        *   公式：$c^{(i)} = \arg\min_k ||x^{(i)} - \mu_k||^2$
3.   **更新阶段**
        *   重新计算每个簇的质心（簇内所有点的均值）
        *   公式：$\mu_k = \frac{1}{|c_k|}\sum_{i\in c_k}x^{(i)}$
4.   **迭代收敛**
        *   重复分配和更新步骤
        *   直到质心变化小于阈值或达到最大迭代次数



下面开始代码实践：

In [ ]:
# 使用肘部法则确定最佳K值

# 初始化存储误差平方和（SSE）和轮廓系数的列表
inertia = [] # 用于存储不同K值对应的误差平方和（SSE）
silhouette_scores = [] # 用于存储不同K值对应的轮廓系数
k_range = range(2, 11) # 测试K值的范围

for k in k_range:
    # 初始化K-means模型，设置簇数量为k
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled) # 使用标准化后的特征数据训练模型
    inertia.append(kmeans.inertia_)
    # 计算并记录当前K值的轮廓系数（评估聚类质量）
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# 绘制肘部法则图
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, 'bo-')
plt.xlabel('簇数量 (K)')
plt.ylabel('SSE (误差平方和)')
plt.title('K-means 肘部法则')

# 绘制轮廓系数图
plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, 'ro-')
plt.xlabel('簇数量 (K)')
plt.ylabel('轮廓系数')
plt.title('轮廓系数评估')
plt.tight_layout()
plt.show()

# 根据轮廓系数选择最佳K值
best_k = k_range[np.argmax(silhouette_scores)]
print(f"最佳簇数量：{best_k} (轮廓系数={max(silhouette_scores):.3f})")

# 使用最佳K值训练模型
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans.fit(X_scaled)
df['KMeans_Cluster'] = kmeans.labels_

# 可视化聚类结果（2D）
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', 
                hue='KMeans_Cluster', palette='viridis', s=80)
plt.title('收入-消费评分聚类结果')
plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)', 
                hue='KMeans_Cluster', palette='viridis', s=80)
plt.title('年龄-消费评分聚类结果')
plt.tight_layout()
plt.show()

# 3D可视化聚类结果
fig = plt.figure(figsize=(14, 10)) # 创建画布
ax = fig.add_subplot(111, projection='3d') # 添加3D子图

# 获取每个簇标签和对应的颜色
clusters = df['KMeans_Cluster'].unique()
colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']

# 遍历每个簇，绘制3D散点
for cluster, color in zip(clusters, colors):
    cluster_data = df[df['KMeans_Cluster'] == cluster]
    # 绘制3D散点：x=年收入，y=消费评分，z=年龄
    ax.scatter(cluster_data['Annual Income (k$)'], 
               cluster_data['Spending Score (1-100)'], 
               cluster_data['Age'], 
               s=60, c=color, label=f'Cluster {cluster}')

# 设置3D坐标轴标签和标题
ax.set_xlabel('年收入 (k$)')
ax.set_ylabel('消费评分')
ax.set_zlabel('年龄')
ax.set_title('K-means 3D聚类结果')
ax.legend()
plt.show()

这些图是用K-means聚类算法分析结果。先通过”肘部法则“和”轮廓系数评估“分析。  
组数越多，混乱程度越低。从肘部法则图中分析看到，分到6组时，混乱程度慢慢下降了，说明6是个合适的临界点  
另一个是轮廓系数，这个值越高说明分组越合理，6组时的系数相对较好，所以最终确定分6组。
接着分析分组后的特征：  

*   ”收入-消费评分“图：不同颜色代表6组，能看到有的组集中在”低收入+低消费“区域，有的在”低收入+高消费“，还有”高收入+低消费“”高收入+低消费“等，比如有的颜色点扎堆在左下角（低收入少花钱），有的在右上角（高收入多花钱），直观体现了收入和消费习惯的关联。
*   ”年龄-消费评分“图：能看出不同年龄层的消费特点，比如有的组集中在”年轻人+高消费“区域，有的在”老年人+低消费“区域，说明年龄和消费积极性有一定关系。
*   3D图（结合年龄、收入、消费评分）：把三个特征放一起看，6组样本各自扎堆，比如有的组是”中年+高收入+中等消费“，有的是”年轻+低收入+高消费“，能更全面地看到不同人群的”年龄、收入、消费“组合特点。

K-means聚类结果揭示了数据中6个具有显著特征差异的子群体，这些群体在”收入-消费“”年龄-消费“及三维复合维度上呈现多样性的关联模式，既体现了”收入、年龄“对消费行为的影响，也暴露了”非单一因素决定消费“的复杂性。

### 3.2 DBSCAN聚类

DBSCAN 是基于密度的无监督聚类算法。它将邻域半径 eps 内点数量超阈值 minPts 的点定义为核心点，相互密度相连的核心点及周边点构成簇，不在任何核心点邻域内的视为噪声点。  
优势在于无需预先指定簇数量，能发现任意形状簇，还能识别噪声点；局限是对 eps 和 minPts 参数敏感，处理密度变化大的数据效果差，计算复杂度较高。它常用于地理信息分析、图像识别、客户细分等场景 。

In [ ]:
# 确定DBSCAN算法的最优eps参数（邻域半径）
# 使用K近邻法：计算每个样本到其第5个最近邻的距离，通过可视化找到合适的eps值

# 初始化最近邻模型，设置寻找5个最近邻
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled) # 用标准化后的特征数据训练模型

# 计算每个样本到其5个最近邻的距离，返回距离矩阵和索引矩阵
distances, _ = nn.kneighbors(X_scaled)
# 提取每个样本到第5个最近邻的距离，并按升序排序
distances = np.sort(distances[:, -1], axis=0)

# 绘制K近邻距离图，用于确定eps参数
plt.figure(figsize=(10, 6))
plt.plot(distances) # 绘制排序后的距离曲线
plt.xlabel('样本排序')
plt.ylabel('第5近邻距离')
plt.title('DBSCAN参数选择')
# 绘制参考水平线（y=0.5），用于确定eps的大致取值
plt.hlines(y=0.5, xmin=0, xmax=200, colors='r', linestyles='dashed')
plt.show()

# 执行DBSCAN聚类
# 初始化DBSCAN模型，设置邻域半径eps=0.5，核心点所需最小样本数为5
dbscan = DBSCAN(eps=0.5, min_samples=5)
# 训练模型并预测聚类结果（-1表示噪声点，其他数值表示不同簇）
dbscan_clusters = dbscan.fit_predict(X_scaled)
df['DBSCAN_Cluster'] = dbscan_clusters

# 统计聚类结果
n_clusters = len(np.unique(dbscan_clusters)) - (1 if -1 in dbscan_clusters else 0)
n_noise = np.sum(dbscan_clusters == -1)
print(f"DBSCAN发现簇数：{n_clusters}")
print(f"噪声点数量：{n_noise}")

# 可视化DBSCAN
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)',
                hue='DBSCAN_Cluster', style='DBSCAN_Cluster',
                palette='coolwarm', s=80)
plt.title('收入-消费评分聚类结果')
plt.tight_layout()
plt.show()

通过 DBSCAN 算法进行聚类分析，先利用 K 近邻距离图确定关键参数 eps，图中蓝色线的快速上升拐点结合红色参考线，选定合适 eps 后，算法识别出 6 个簇（群体） 与 60 个噪声点  
在 “收入 - 消费评分” 散点图里，不同形状、颜色代表的簇呈现出多样分布：部分簇在 “收入与消费匹配” 区域（如高收入高消费、低收入低消费 ）聚集，部分簇在 “收入与消费错位” 区域（如高收入低消费、低收入高消费 ）分布，还有零散噪声点  
两种算法均识别出6个核心群体，验证了数据中群体划分的客观性，展现出不同收入-消费行为模式的人群特征。

### 3.3 聚类结果分析

聚类算法输出的簇标签仅仅是一串数字符号，真正的分析始于这些符号背后的意义挖掘。我们站在算法与现实世界的交界处：一方面需要验证数学聚类的有效性，另一方面必须要将抽象的数据结构转化为可理解的群体画像。通过统计指标确保簇内同质性和簇间异质性。

In [ ]:
# 群体特征分析
print('K-means聚类群体特征')
kmeans_profile = df.groupby('KMeans_Cluster').agg({
    'Age': ['mean', 'std'],
    'Annual Income (k$)': ['mean', 'std'],
    'Spending Score (1-100)': ['mean', 'std'],
    'Gender': lambda x: x.mode()[0]
}).reset_index()
display(kmeans_profile)

print("\nDBSCAN聚类群体特征：")
dbscan_profile = df[df['DBSCAN_Cluster'] != -1].groupby('DBSCAN_Cluster').agg({
    'Age': ['mean', 'std'],
    'Annual Income (k$)': ['mean', 'std'],
    'Spending Score (1-100)': ['mean', 'std'],
    'Gender': lambda x: x.mode()[0]
}).reset_index()
display(dbscan_profile)

# 轮廓系数评估
kmeans_score = silhouette_score(X_scaled, df['KMeans_Cluster'])
dbscan_score = silhouette_score(X_scaled[dbscan_clusters != -1],
                                dbscan_clusters[dbscan_clusters != -1])
print(f"\nK-means群体划分业务含义：")
print("Cluster 0: 年轻高消费群体（18-30岁），中等收入，高消费")
print("Cluster 1: 高收入低消费群体（各年龄段，高收入，低消费）")
print("Cluster 2: 中等收入中等消费群体（30-50岁，中等收入，中等消费）")
print("Cluster 3: 低收入高消费群体（各年龄段，低收入，高消费）")
print("Cluster 4: 高收入低消费群体（黄金客户，高收入，高消费）")

print("\nDBSCAN群体划分业务含义：")
print("Cluster 0: 核心消费群体（中等收入，中等消费）")
print("Cluster 1: 高消费年轻群体（中等收入，高消费）")
print("Cluster 2: 高收入保守群体（高收入，低消费）")
print("噪声点: 特殊消费模式客户")

# 营销策略建议
print("\n营销策略建议：")
print("1. 针对高收入高消费群体（Cluster 4）: 提供VIP服务和高端商品")
print("2. 针对年轻高消费群体（Cluster 0）: 推出时尚潮流产品和社交媒体营销")
print("3. 针对高收入低消费群体（Cluster 1）: 提供投资理财服务和品质生活体验")
print("4. 针对低收入高消费群体（Cluster 3）: 提供分期付款和促销活动")

## 4. 总结

本实训方案系统地展示了聚类算法在模式识别与群体划分中的实际应用，通过K-means和DBSCAN算法对商场客户消费数据进行深度挖掘。方案从算法原理介绍入手，结合具体项目场景，完整呈现了数据探索、特征工程、模型构建到结果可视化的全流程，关键价值在于：  

1.   **技术实践**：通过肘部法则确定最佳聚类数
2.   **业务洞察**：识别出5类特征鲜明的消费群体
3.   **方法论验证**：对比验证了划分式聚类（K-means）与密度聚类（DBSCAN）的适用场景与局限
4.   **决策支持**：基于聚类结果提出差异化营销策略，将技术成果转化为商业价值  

